# Module 6.8 — Code Review & Pythonic Excellence

### Python Mastery · Track 6: Testing, Tooling & DevOps

Working code is only the start; code is read far more often than it is written, so clarity and maintainability matter enormously. This module covers writing readable, idiomatic Python, conducting effective code reviews, and practical refactoring techniques that improve code without changing its behaviour.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it. Most examples show a weaker version and a clearer rewrite, with the behaviour verified to be identical.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Recognise and write "Pythonic" code that reads clearly.
2. Apply common refactoring recipes safely.
3. Simplify conditionals and extract well-named functions.
4. Conduct a constructive code review against a checklist.
5. Articulate principles for maintainable code and team standards.

**Prerequisites:** Tracks 1 to 5, and Modules 6.1 to 6.7.

---

## Part 1 · Writing Pythonic Code

"Pythonic" code uses the language's idioms to express intent directly, rather than translating habits from other languages. It is usually shorter, clearer, and less error-prone. A few common improvements: iterate directly over items rather than indices, use truthiness and comprehensions, and let built-ins do the work. The cell below contrasts non-idiomatic code with its Pythonic equivalent, confirming both produce the same result.

In [ ]:
names = ["asha", "ben", "chen"]

# Non-idiomatic: indexing manually, building a list step by step.
result_a = []
for i in range(len(names)):
    result_a.append(names[i].upper())

# Pythonic: iterate over items directly, use a comprehension.
result_b = [name.upper() for name in names]

print("manual:   ", result_a)
print("pythonic: ", result_b)
print("identical:", result_a == result_b)

In [ ]:
# Non-idiomatic checks versus Pythonic ones.
items = []

# Verbose emptiness check.
if len(items) == 0:
    verbose = "empty"
else:
    verbose = "has items"

# Pythonic: an empty list is falsy.
pythonic = "empty" if not items else "has items"

print(verbose, "==", pythonic)

# enumerate instead of manual counters; zip to pair sequences.
letters = ["a", "b", "c"]
numbers = [1, 2, 3]
print("paired:", [f"{l}{n}" for l, n in zip(letters, numbers)])
print("indexed:", [f"{i}:{l}" for i, l in enumerate(letters)])

## Part 2 · Extracting Functions

A long function that does several things is hard to read, test, and reuse. **Extracting** each distinct step into a well-named function turns a wall of code into a readable summary of what happens. The names become documentation, and each small function can be tested on its own.

In [ ]:
# Before: one function doing validation, calculation, and formatting all at once.
def process_order_before(items):
    # validate
    for item in items:
        if item["price"] < 0:
            raise ValueError("negative price")
    # calculate
    subtotal = sum(item["price"] * item["qty"] for item in items)
    tax = subtotal * 0.18
    total = subtotal + tax
    # format
    return f"Subtotal: {subtotal:.2f}, Tax: {tax:.2f}, Total: {total:.2f}"

# After: each step is its own named function; the top-level reads like a summary.
def validate(items):
    for item in items:
        if item["price"] < 0:
            raise ValueError("negative price")

def compute_total(items):
    subtotal = sum(item["price"] * item["qty"] for item in items)
    tax = subtotal * 0.18
    return subtotal, tax, subtotal + tax

def format_summary(subtotal, tax, total):
    return f"Subtotal: {subtotal:.2f}, Tax: {tax:.2f}, Total: {total:.2f}"

def process_order_after(items):
    validate(items)
    subtotal, tax, total = compute_total(items)
    return format_summary(subtotal, tax, total)

order = [{"price": 100, "qty": 2}, {"price": 50, "qty": 1}]
print("before:", process_order_before(order))
print("after: ", process_order_after(order))
print("same result:", process_order_before(order) == process_order_after(order))

## Part 3 · Simplifying Conditionals

Deeply nested conditionals are hard to follow. Two reliable techniques flatten them: **guard clauses** return early for the exceptional cases, leaving the main logic unindented; and replacing a chain of `if`/`elif` on a value with a **dictionary lookup** removes branching entirely where it fits.

In [ ]:
# Before: nested conditionals, hard to scan.
def describe_before(user):
    if user is not None:
        if user.get("active"):
            if user.get("age", 0) >= 18:
                return "active adult"
            else:
                return "active minor"
        else:
            return "inactive"
    else:
        return "no user"

# After: guard clauses handle the exceptional cases first, flattening the logic.
def describe_after(user):
    if user is None:
        return "no user"
    if not user.get("active"):
        return "inactive"
    if user.get("age", 0) < 18:
        return "active minor"
    return "active adult"

cases = [None, {"active": False}, {"active": True, "age": 15}, {"active": True, "age": 30}]
for case in cases:
    print(describe_before(case), "==", describe_after(case))

In [ ]:
# Replacing an if/elif chain with a dictionary lookup.
def grade_before(score):
    if score >= 90:
        return "A"
    elif score >= 80:
        return "B"
    elif score >= 70:
        return "C"
    else:
        return "F"

import bisect
def grade_after(score):
    # Boundaries and bands; bisect picks the right band (see Module 2.5).
    boundaries = [70, 80, 90]
    bands = ["F", "C", "B", "A"]
    return bands[bisect.bisect_right(boundaries, score)]

for s in [95, 85, 72, 50]:
    print(s, grade_before(s), grade_after(s))

## Part 4 · Conducting a Code Review

A code review is a constructive conversation about a change, not a test to pass. Good reviews improve the code and spread knowledge across the team. A practical checklist:

- **Correctness:** does it do what it claims, including edge cases?
- **Clarity:** can the next reader understand it without the author present?
- **Tests:** are there tests, and do they cover the important cases?
- **Naming:** are names accurate and consistent?
- **Simplicity:** is anything more complex than it needs to be?
- **Safety:** is input validated, and are there security or performance concerns?

Feedback should be specific, kind, and focused on the code, not the person. Distinguish required changes from optional suggestions, and explain the reasoning so the author learns. The example below shows the spirit of review comments applied to a snippet.

In [ ]:
# A snippet under review, with the kinds of comments a reviewer might leave.
def get_data(l):                          # comment: rename 'l'; it is unclear and looks like 1
    d = {}                                # comment: 'd' is vague; name it for what it holds
    for x in l:                           # comment: 'x' could be 'record'
        d[x[0]] = x[1]                    # comment: index access is unclear; unpack instead
    return d

# A revised version addressing the comments.
def build_lookup(records):
    """Build a name-to-value mapping from (name, value) pairs."""
    return {name: value for name, value in records}

pairs = [("a", 1), ("b", 2)]
print("original:", get_data(pairs))
print("revised: ", build_lookup(pairs))
print("same behaviour:", get_data(pairs) == build_lookup(pairs))

## Part 5 · Principles for Maintainable Code

A few principles, drawn from this whole track, guide code that lasts:

- **Readability counts:** optimise for the next reader, who is often you.
- **Keep functions small and focused:** one job each, with a clear name.
- **Do not repeat yourself:** extract shared logic rather than copying it.
- **Make the common case simple:** handle edge cases without obscuring the main path.
- **Let tools enforce consistency:** formatting, linting, and types free reviewers to focus on substance.
- **Write tests:** they document intent and protect against regressions.

Teams codify such principles into a shared standard, often a short style document plus the automated tools from earlier modules, so the codebase stays coherent as it grows and as people come and go.

In [ ]:
# A short demonstration of "do not repeat yourself": collapsing duplicated logic.
# Before: the same formatting repeated for each field.
def report_before(user):
    line1 = "Name: " + user["name"].strip().title()
    line2 = "City: " + user["city"].strip().title()
    line3 = "Role: " + user["role"].strip().title()
    return "\n".join([line1, line2, line3])

# After: one helper, applied uniformly; adding a field is now trivial.
def clean(text):
    return text.strip().title()

def report_after(user):
    fields = ["name", "city", "role"]
    return "\n".join(f"{field.title()}: {clean(user[field])}" for field in fields)

user = {"name": " asha ", "city": "pune", "role": "engineer"}
print(report_before(user))
print("---")
print(report_after(user))

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — Iterating Pythonically (Easy)

In [ ]:
words = ["red", "green", "blue"]

# Manual indexing.
out_a = []
for i in range(len(words)):
    out_a.append((i, words[i]))

# Pythonic with enumerate.
out_b = list(enumerate(words))

print(out_a == out_b, out_b)

### Example 2 — Using truthiness (Easy)

In [ ]:
def status(items):
    # Pythonic: rely on the falsiness of an empty collection.
    return "empty" if not items else f"{len(items)} items"

print(status([]))
print(status([1, 2, 3]))

### Example 3 — Guard clauses (Medium)

In [ ]:
def can_withdraw_before(account, amount):
    if account is not None:
        if account["active"]:
            if amount <= account["balance"]:
                return True
    return False

def can_withdraw_after(account, amount):
    if account is None:
        return False
    if not account["active"]:
        return False
    return amount <= account["balance"]

acc = {"active": True, "balance": 100}
for amt in [50, 200]:
    print(can_withdraw_before(acc, amt), can_withdraw_after(acc, amt))

### Example 4 — Extracting a function (Medium)

In [ ]:
# Before: a function mixing two responsibilities.
def summarise_before(numbers):
    cleaned = [n for n in numbers if n is not None]
    return sum(cleaned) / len(cleaned) if cleaned else 0

# After: separate the cleaning step, making each part testable.
def remove_missing(numbers):
    return [n for n in numbers if n is not None]

def average(numbers):
    values = remove_missing(numbers)
    return sum(values) / len(values) if values else 0

data = [10, None, 20, None, 30]
print(summarise_before(data), "==", average(data))

### Example 5 — Replacing branching with a lookup (Difficult)

In [ ]:
# Before: a chain of conditionals mapping a command to an action.
def handle_before(command):
    if command == "start":
        return "starting"
    elif command == "stop":
        return "stopping"
    elif command == "pause":
        return "pausing"
    else:
        return "unknown"

# After: a dictionary of handlers; adding a command is a one-line change.
def handle_after(command):
    actions = {
        "start": lambda: "starting",
        "stop": lambda: "stopping",
        "pause": lambda: "pausing",
    }
    handler = actions.get(command, lambda: "unknown")
    return handler()

for cmd in ["start", "pause", "fly"]:
    print(cmd, handle_before(cmd), handle_after(cmd))

### Example 6 — A reviewed and refactored function (Difficult)

In [ ]:
# A function as first submitted: unclear names, repetition, deep nesting.
def proc(d):
    r = []
    for k in d:
        if d[k] != None:
            if d[k] > 0:
                r.append((k, d[k] * 2))
    return r

# After review: clear names, a guard, and a comprehension.
def double_positive_values(data):
    """Return (key, doubled value) pairs for positive, non-missing values."""
    return [(key, value * 2) for key, value in data.items()
            if value is not None and value > 0]

sample = {"a": 5, "b": None, "c": -3, "d": 10}
print("original:", proc(sample))
print("refactored:", double_positive_values(sample))
print("same:", proc(sample) == double_positive_values(sample))

---

## Exercises

**Exercise 1 (Easy).** Rewrite this loop Pythonically using a comprehension, and confirm the result is unchanged: `result = []` then `for i in range(len(nums)): result.append(nums[i] * 10)`.

In [ ]:
nums = [1, 2, 3, 4]
# Your solution here


**Exercise 2 (Easy).** Rewrite a verbose emptiness check `if len(data) == 0: ... else: ...` to use truthiness, for a function returning `"none"` or `"some"`.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Rewrite the nested function below using guard clauses so the happy path is unindented, keeping the same behaviour.

In [ ]:
def check_before(order):
    if order is not None:
        if order.get("paid"):
            if order.get("items"):
                return "ready to ship"
            else:
                return "no items"
        else:
            return "unpaid"
    else:
        return "no order"
# Write check_after below and confirm it matches check_before on a few inputs.


**Exercise 4 (Medium).** Extract a helper function from this code so the main function reads clearly: a function that takes a list of strings and returns those that are non-empty after stripping whitespace, uppercased.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Refactor the poorly written function below: give meaningful names, remove the deep nesting with a comprehension and a guard, and verify the behaviour is identical.

In [ ]:
def f(x):
    o = []
    for i in x:
        if i != None:
            if len(i) > 3:
                o.append(i.upper())
    return o
# Write a clean version and compare on a sample list of strings (some None, some short).


---

## Solutions

**Solution 1**

In [ ]:
nums = [1, 2, 3, 4]
result = [n * 10 for n in nums]
print(result)

**Solution 2**

In [ ]:
def describe(data):
    return "none" if not data else "some"

print(describe([]), describe([1]))

**Solution 3**

In [ ]:
def check_before(order):
    if order is not None:
        if order.get("paid"):
            if order.get("items"):
                return "ready to ship"
            else:
                return "no items"
        else:
            return "unpaid"
    else:
        return "no order"

def check_after(order):
    if order is None:
        return "no order"
    if not order.get("paid"):
        return "unpaid"
    if not order.get("items"):
        return "no items"
    return "ready to ship"

cases = [None, {"paid": False}, {"paid": True, "items": []}, {"paid": True, "items": ["x"]}]
for c in cases:
    print(check_before(c), "==", check_after(c))

**Solution 4**

In [ ]:
def clean_nonempty(strings):
    """Return stripped, uppercased versions of the non-empty strings."""
    return [s.strip().upper() for s in strings if s.strip()]

print(clean_nonempty(["  hello ", "   ", "world", ""]))

**Solution 5**

In [ ]:
def f(x):
    o = []
    for i in x:
        if i != None:
            if len(i) > 3:
                o.append(i.upper())
    return o

def long_words_upper(words):
    """Return the uppercase form of non-missing words longer than three letters."""
    return [word.upper() for word in words if word is not None and len(word) > 3]

sample = ["hi", None, "hello", "ok", "python"]
print(f(sample))
print(long_words_upper(sample))
print("same:", f(sample) == long_words_upper(sample))

---

## Summary and Key Points

- Pythonic code uses the language's idioms: iterate over items not indices, rely on truthiness, use comprehensions, `enumerate`, and `zip`, and let built-ins do the work.
- Extract distinct steps into small, well-named functions; the names document the code and each piece becomes testable in isolation.
- Flatten nested conditionals with guard clauses that return early, and replace value-based `if`/`elif` chains with dictionary lookups where they fit.
- A code review is a constructive conversation: check correctness, clarity, tests, naming, simplicity, and safety, and give specific, kind, reasoned feedback about the code.
- Maintainable code favours readability, small focused functions, no repetition, a simple common path, automated consistency, and tests; teams codify these as a shared standard.

### Track 6 complete

This concludes Track 6, Testing, Tooling & DevOps. You can now write and automate tests, debug and profile, add and check type hints, enforce quality with linters and formatters, package and distribute code, set up continuous integration and containers, write secure code, and review and refactor for clarity. Track 7 turns to performance, internals, and extending Python with C.